In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## EDA

In [1]:
import os
dataset_dir = 'D:\model_descriptor\ML_MAGA\project\dataset'

os.listdir(dataset_dir)

['test', 'test.csv', 'train']

In [2]:
training_dir = f"{dataset_dir}/train"
files = os.listdir(training_dir)

In [3]:
is_json = lambda x: x.endswith("json")
is_img = lambda x: x.endswith("jpg")

In [4]:
import json
coco_json = next(filter(is_json, files))

In [5]:
with open(f"{training_dir}/{coco_json}") as coco_file:
    coco_json = json.load(coco_file)
    # print(coco_json.keys())
    coco_annotations = coco_json.get("annotations")

In [6]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [7]:
categories = coco_json['categories']
category_map = {cat['id']: cat['name'] for cat in categories}


category_counts = {cat['id']: 0 for cat in categories}

for ann in coco_json['annotations']:
    category_counts[ann['category_id']] += 1

In [8]:
categories

[{'id': 0, 'name': 'football-objects', 'supercategory': 'none'},
 {'id': 1, 'name': 'ball', 'supercategory': 'football-objects'},
 {'id': 2, 'name': 'coach', 'supercategory': 'football-objects'},
 {'id': 3, 'name': 'goalkeeper', 'supercategory': 'football-objects'},
 {'id': 4, 'name': 'player', 'supercategory': 'football-objects'},
 {'id': 5, 'name': 'referee', 'supercategory': 'football-objects'}]

In [9]:
class_distribution = pd.DataFrame([
    {'category_id': cat_id, 'category_name': category_map[cat_id], 'count': count} 
    for cat_id, count in category_counts.items()
])

# Sort by count
class_distribution = class_distribution.sort_values('count', ascending=False)

In [10]:
coco_json.keys()

dict_keys(['info', 'licenses', 'categories', 'images', 'annotations'])

In [11]:
coco_json['annotations'][0].keys()

dict_keys(['id', 'image_id', 'category_id', 'bbox', 'area', 'segmentation', 'iscrowd'])

In [69]:
coco_json['images'][0].keys()

dict_keys(['id', 'license', 'file_name', 'height', 'width', 'date_captured', 'extra'])

In [ ]:
coco_json['images']

In [12]:
import cv2

def draw_bounding_boxes(coco_json, image_path, image_id):

    file_name = None
    for i in coco_json['images']:
        if i["id"] == image_id:
            file_name = i['file_name']
            break

    if file_name is None:
        raise FileNotFoundError(f"Не удалось найти id изображения: {image_id}")

    # Путь к изображению
    image_path += file_name

    image_boxes = []
    for i in coco_json['annotations']:
        if (i["image_id"] == image_id):
            image_boxes.append(i)

    # Загрузка изображения
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Не удалось загрузить изображение: {image_path}")

    for i in image_boxes:

        bbox = i["bbox"]
        
        # Рисуем прямоугольник
        x_min, y_min, x_max, y_max = map(int, bbox)
        x_max += x_min
        y_max += y_min
        cv2.rectangle(img, (x_min, y_min), (x_max, y_max), color=(0, 255, 0), thickness=2)

    cv2.imshow("Image with BBox", img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

In [ ]:
image_path = "D:\\model_descriptor\\ML_MAGA\\project\\dataset\\train\\"

draw_bounding_boxes(coco_json, image_path, 200)

# Данные

In [1]:
import random
from collections import defaultdict

# Получаем все image_id -> [annotations]
img_to_anns = defaultdict(list)
for ann in coco_json['annotations']:
    img_to_anns[ann['image_id']].append(ann)

image_ids = list(img_to_anns.keys())
random.seed(42)
random.shuffle(image_ids)

split = int(0.8 * len(image_ids))
train_ids = set(image_ids[:split])
val_ids = set(image_ids[split:])

NameError: name 'coco_json' is not defined

In [86]:
import os
import json
from pathlib import Path

# Настройки
OUTPUT_DIR = Path("coco_yolo")
IMG_SRC_DIR = Path("D:\\model_descriptor\\ML_MAGA\\project\\dataset\\train")  # где лежат .jpg/.png из file_name

# Создаём структуру
for split in ['train', 'val']:
    (OUTPUT_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

# Вспомогательная функция: COCO bbox → YOLO
def coco_to_yolo(bbox, img_w, img_h):
    # bbox = [x, y, w, h] — в COCO: top-left corner + ширина/высота
    x, y, w, h = bbox
    center_x = (x + w / 2) / img_w
    center_y = (y + h / 2) / img_h
    norm_w = w / img_w
    norm_h = h / img_h
    return center_x, center_y, norm_w, norm_h

# Маппинг category_id → class_id (0-based)
cat_id_to_idx = {cat['id']: i for i, cat in enumerate(coco_json['categories'])}

# Проходим по изображениям
for img_info in coco_json['images']:
    img_id = img_info['id']
    file_name = img_info['file_name']
    width = img_info['width']
    height = img_info['height']
    
    # Определяем, в каком сплите изображение
    split = 'train' if img_id in train_ids else 'val'
    
    # Копируем изображение (или создаём симлинк)
    src = IMG_SRC_DIR / file_name
    dst_img = OUTPUT_DIR / 'images' / split / file_name
    if not dst_img.exists():
        # Можно shutil.copy, или os.symlink для экономии места
        import shutil
        shutil.copy(src, dst_img)
    
    # Пишем аннотации в labels/
    dst_lbl = OUTPUT_DIR / 'labels' / split / (Path(file_name).stem + '.txt')
    with open(dst_lbl, 'w') as f:
        for ann in img_to_anns[img_id]:
            cat_id = ann['category_id']
            class_id = cat_id_to_idx[cat_id]
            bbox = ann['bbox']
            cx, cy, w, h = coco_to_yolo(bbox, width, height)
            f.write(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")

In [2]:
import json
import os
import shutil
import yaml
from sklearn.model_selection import train_test_split
import cv2
from tqdm import tqdm

# 1. Загружаем JSON файл с аннотациями
with open('D:\\model_descriptor\\ML_MAGA\\project\\dataset\\train\\_annotations.coco.json', 'r') as f:
    coco_json = json.load(f)

print(f"Загружено {len(coco_json['images'])} изображений")
print(f"Загружено {len(coco_json['annotations'])} аннотаций")
print(f"Классы: {[cat['name'] for cat in coco_json['categories']]}")

# 2. Создаем директории для YOLOv8
def create_yolo_structure(base_path='./yolo_football'):
    """Создает структуру папок для YOLOv8"""
    directories = [
        f'{base_path}/train/images',
        f'{base_path}/train/labels',
        f'{base_path}/val/images',
        f'{base_path}/val/labels',
        f'{base_path}/test/images',
        f'{base_path}/test/labels',
    ]
    
    for dir_path in directories:
        os.makedirs(dir_path, exist_ok=True)
    
    print(f"Создана структура папок в {base_path}")
    return base_path

# 3. Функция для конвертации COCO в YOLO формат
def convert_coco_to_yolo(img_info, annotations, output_dir, split='train'):
    """
    Конвертирует одно изображение из COCO в YOLO формат
    
    Args:
        img_info: информация об изображении из coco_json['images']
        annotations: список аннотаций для этого изображения
        output_dir: базовая директория вывода
        split: 'train' или 'val'
    """
    # Путь к исходному изображению
    src_image_path = f"D:\\model_descriptor\\ML_MAGA\\project\\dataset\\{split}/{img_info['file_name']}"
    
    if not os.path.exists(src_image_path):
        # Пробуем найти в train если не нашли в указанном split
        src_image_path = f"D:\\model_descriptor\\ML_MAGA\\project\\dataset\\train\\{img_info['file_name']}"
    
    if not os.path.exists(src_image_path):
        print(f"Изображение не найдено: {img_info['file_name']}")
        return False
    
    # Копируем изображение
    dst_image_path = f"{output_dir}/{split}/images/{img_info['file_name']}"
    shutil.copy2(src_image_path, dst_image_path)
    
    # Создаем YOLO аннотацию
    img_width = img_info['width']
    img_height = img_info['height']
    
    # Создаем файл с метками
    label_filename = os.path.splitext(img_info['file_name'])[0] + '.txt'
    label_path = f"{output_dir}/{split}/labels/{label_filename}"
    
    with open(label_path, 'w') as f:
        for ann in annotations:
            # COCO формат: [x_min, y_min, width, height]
            x_min, y_min, width, height = ann['bbox']
            
            # Конвертируем в YOLO формат: [class_id, x_center, y_center, width, height]
            # Все значения нормализуем относительно размеров изображения
            x_center = (x_min + width / 2) / img_width
            y_center = (y_min + height / 2) / img_height
            width_norm = width / img_width
            height_norm = height / img_height
            
            # COCO category_id обычно начинается с 1, делаем 0-based
            class_id = ann['category_id']
            
            # Записываем в файл
            f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {width_norm:.6f} {height_norm:.6f}\n")
    
    return True

# 4. Основная функция подготовки
def prepare_yolo_dataset(test_size=0.2, random_state=42):
    """Основная функция подготовки данных для YOLOv8"""
    
    # Создаем структуру папок
    base_path = create_yolo_structure('./yolo_football_dataset')
    
    # Создаем mapping для быстрого поиска аннотаций
    annotations_by_image = {}
    for ann in coco_json['annotations']:
        img_id = ann['image_id']
        if img_id not in annotations_by_image:
            annotations_by_image[img_id] = []
        annotations_by_image[img_id].append(ann)
    
    # Создаем словарь изображений
    images_by_id = {img['id']: img for img in coco_json['images']}
    
    # Разделяем на train/val
    image_ids = list(images_by_id.keys())
    train_ids, val_ids = train_test_split(
        image_ids, 
        test_size=test_size, 
        random_state=random_state
    )
    
    print(f"\nРазделение данных:")
    print(f"  Train: {len(train_ids)} изображений")
    print(f"  Val:   {len(val_ids)} изображений")
    
    # Обрабатываем train данные
    print("\nОбработка train данных...")
    train_success = 0
    for img_id in tqdm(train_ids):
        img_info = images_by_id[img_id]
        annotations = annotations_by_image.get(img_id, [])
        
        if convert_coco_to_yolo(img_info, annotations, base_path, split='train'):
            train_success += 1
    
    # Обрабатываем val данные
    print("\nОбработка val данных...")
    val_success = 0
    for img_id in tqdm(val_ids):
        img_info = images_by_id[img_id]
        annotations = annotations_by_image.get(img_id, [])
        
        if convert_coco_to_yolo(img_info, annotations, base_path, split='val'):
            val_success += 1
    
    print(f"\nОбработано успешно:")
    print(f"   Train: {train_success}/{len(train_ids)}")
    print(f"   Val:   {val_success}/{len(val_ids)}")
    
    # 5. Создаем dataset.yaml
    create_dataset_yaml(coco_json, base_path)
    
    return base_path

def create_dataset_yaml(coco_data, base_path):
    """Создает dataset.yaml файл для YOLOv8"""
    
    # Получаем имена классов в правильном порядке
    categories_sorted = sorted(coco_data['categories'], key=lambda x: x['id'])
    class_names = [cat['name'] for cat in categories_sorted]
    
    # Создаем конфигурацию
    config = {
        'path': os.path.abspath(base_path),  # Абсолютный путь
        'train': 'train/images',  # Относительно path
        'val': 'val/images',      # Относительно path
        
        # Количество классов
        'nc': len(class_names),
        
        # Имена классов
        'names': class_names
    }
    
    # Сохраняем YAML файл
    yaml_path = os.path.join(base_path, 'dataset.yaml')
    with open(yaml_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False)
    
    print(f"\nСоздан файл конфигурации: {yaml_path}")
    print(f"   Classes: {class_names}")
    
    return yaml_path

# 6. Функция для проверки конвертации
def verify_conversion(sample_image_id):
    """Проверяет правильность конвертации на примере"""
    
    # Находим изображение
    img_info = None
    for img in coco_json['images']:
        if img['id'] == sample_image_id:
            img_info = img
            break
    
    if not img_info:
        print(f"Image ID {sample_image_id} not found")
        return
    
    # Получаем аннотации
    annotations = []
    for ann in coco_json['annotations']:
        if ann['image_id'] == sample_image_id:
            annotations.append(ann)
    
    # Загружаем исходное изображение
    src_path = f"D:\\model_descriptor\\ML_MAGA\\project\\dataset\\train\\{img_info['file_name']}"
    img = cv2.imread(src_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Рисуем bounding boxes из COCO
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    
    # 1. Оригинальные COCO bboxes
    img_coco = img_rgb.copy()
    for ann in annotations:
        x_min, y_min, width, height = map(int, ann['bbox'])
        x_max = x_min + width
        y_max = y_min + height
        
        # Рисуем прямоугольник
        cv2.rectangle(img_coco, (x_min, y_min), (x_max, y_max), color=(0, 255, 0), thickness=2)
        
        # Добавляем текст с классом
        class_name = next((cat['name'] for cat in coco_json['categories'] 
                          if cat['id'] == ann['category_id']), 'unknown')
        cv2.putText(img_coco, class_name, (x_min, y_min - 10), 
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    
    axes[0].imshow(img_coco)
    axes[0].set_title('Original COCO Annotations')
    axes[0].axis('off')
    
    # 2. Загружаем YOLO аннотацию и рисуем
    # (если уже сконвертировано)
    yolo_label_path = f"./yolo_football_dataset/train/labels/{os.path.splitext(img_info['file_name'])[0]}.txt"
    
    if os.path.exists(yolo_label_path):
        img_yolo = img_rgb.copy()
        img_height, img_width = img_rgb.shape[:2]
        
        with open(yolo_label_path, 'r') as f:
            lines = f.readlines()
        
        for line in lines:
            parts = line.strip().split()
            if len(parts) == 5:
                class_id, x_center, y_center, w_norm, h_norm = map(float, parts)
                
                # Конвертируем обратно в пиксельные координаты
                x_center_px = x_center * img_width
                y_center_px = y_center * img_height
                w_px = w_norm * img_width
                h_px = h_norm * img_height
                
                x_min = int(x_center_px - w_px / 2)
                y_min = int(y_center_px - h_px / 2)
                x_max = int(x_center_px + w_px / 2)
                y_max = int(y_center_px + h_px / 2)
                
                # Рисуем
                cv2.rectangle(img_yolo, (x_min, y_min), (x_max, y_max), color=(255, 0, 0), thickness=2)
                
                # Получаем имя класса
                class_name = coco_json['categories'][int(class_id)]['name']
                cv2.putText(img_yolo, class_name, (x_min, y_min - 10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
        
        axes[1].imshow(img_yolo)
        axes[1].set_title('YOLO Format Annotations')
        axes[1].axis('off')
    else:
        axes[1].text(0.5, 0.5, 'YOLO file not found yet\nRun prepare_yolo_dataset() first', 
                    ha='center', va='center', fontsize=12)
        axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

# 7. Запуск подготовки данных
if __name__ == "__main__":
    print("=" * 60)
    print("ПОДГОТОВКА ДАННЫХ ДЛЯ YOLOv8")
    print("=" * 60)
    
    # Подготавливаем данные
    dataset_path = prepare_yolo_dataset(test_size=0.2, random_state=42)
    
    print("\n" + "=" * 60)
    print("ДАННЫЕ ГОТОВЫ!")
    print("=" * 60)
    
    print(f"\nСтруктура датасета:")
    print(f"   {dataset_path}/")
    print(f"   ├── train/")
    print(f"   │   ├── images/    # {len(os.listdir(f'{dataset_path}/train/images'))} файлов")
    print(f"   │   └── labels/    # {len(os.listdir(f'{dataset_path}/train/labels'))} файлов")
    print(f"   ├── val/")
    print(f"   │   ├── images/    # {len(os.listdir(f'{dataset_path}/val/images'))} файлов")
    print(f"   │   └── labels/    # {len(os.listdir(f'{dataset_path}/val/labels'))} файлов")
    print(f"   └── dataset.yaml   # конфигурационный файл")
    
    print(f"\nДля обучения запустите:")
    print(f"   from ultralytics import YOLO")
    print(f"   model = YOLO('yolov8s.pt')")
    print(f"   model.train(data='{dataset_path}/dataset.yaml', epochs=100, imgsz=640)")
    
    # Проверяем на примере
    print(f"\nПроверка конвертации (первые 3 изображения):")
    sample_ids = list(coco_json['images'])[:3]
    for img_info in sample_ids:
        verify_conversion(img_info['id'])

Загружено 800 изображений
Загружено 14804 аннотаций
Классы: ['football-objects', 'ball', 'coach', 'goalkeeper', 'player', 'referee']
ПОДГОТОВКА ДАННЫХ ДЛЯ YOLOv8
Создана структура папок в ./yolo_football_dataset

Разделение данных:
  Train: 640 изображений
  Val:   160 изображений

Обработка train данных...


100%|██████████| 640/640 [00:01<00:00, 467.16it/s]



Обработка val данных...


100%|██████████| 160/160 [00:00<00:00, 506.11it/s]



Обработано успешно:
   Train: 640/640
   Val:   160/160

Создан файл конфигурации: ./yolo_football_dataset\dataset.yaml
   Classes: ['football-objects', 'ball', 'coach', 'goalkeeper', 'player', 'referee']

ДАННЫЕ ГОТОВЫ!

Структура датасета:
   ./yolo_football_dataset/
   ├── train/
   │   ├── images/    # 640 файлов
   │   └── labels/    # 640 файлов
   ├── val/
   │   ├── images/    # 160 файлов
   │   └── labels/    # 160 файлов
   └── dataset.yaml   # конфигурационный файл

Для обучения запустите:
   from ultralytics import YOLO
   model = YOLO('yolov8s.pt')
   model.train(data='./yolo_football_dataset/dataset.yaml', epochs=100, imgsz=640)

Проверка конвертации (первые 3 изображения):


NameError: name 'plt' is not defined

In [1]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')
results = model.train(data='D:\model_descriptor\ML_MAGA\project\yolo_football_dataset\dataset.yaml', epochs=16, imgsz=640, model='yolov8s.pt', workers=0, cache=False)

Ultralytics 8.3.235  Python-3.11.9 torch-2.8.0+cu129 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\model_descriptor\ML_MAGA\project\yolo_football_dataset\dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=16, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train11, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, o

In [10]:
result = model.predict('D:\model_descriptor\ML_MAGA\project\dataset\\test\\frame_001064_png.rf.fb03f13c3d2a3b52ac100f10a4948337.jpg', save=True)


image 1/1 D:\model_descriptor\ML_MAGA\project\dataset\test\frame_001064_png.rf.fb03f13c3d2a3b52ac100f10a4948337.jpg: 384x640 1 ball, 16 players, 1 referee, 56.3ms
Speed: 3.1ms preprocess, 56.3ms inference, 4.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to D:\model_descriptor\ML_MAGA\project\runs\detect\predict


In [15]:
result[0].boxes

ultralytics.engine.results.Boxes object with attributes:

cls: tensor([4., 5., 4., 4., 4., 4., 4., 4., 4., 4., 4., 4., 4., 4., 4., 4., 4., 1.], device='cuda:0')
conf: tensor([0.9007, 0.9004, 0.8843, 0.8813, 0.8810, 0.8773, 0.8699, 0.8612, 0.8610, 0.8590, 0.8584, 0.8577, 0.8511, 0.8343, 0.8167, 0.8138, 0.7478, 0.5967], device='cuda:0')
data: tensor([[6.7837e+01, 4.0793e+02, 1.2157e+02, 5.1567e+02, 9.0071e-01, 4.0000e+00],
        [9.6156e+02, 3.7964e+02, 9.9453e+02, 4.7723e+02, 9.0038e-01, 5.0000e+00],
        [5.1998e+02, 4.0563e+02, 5.6695e+02, 5.1440e+02, 8.8426e-01, 4.0000e+00],
        [2.8011e+02, 4.7295e+02, 3.2722e+02, 5.9284e+02, 8.8135e-01, 4.0000e+00],
        [8.2452e+02, 2.8119e+02, 8.7564e+02, 3.6629e+02, 8.8097e-01, 4.0000e+00],
        [7.3981e+02, 5.0870e+02, 7.8895e+02, 6.1860e+02, 8.7731e-01, 4.0000e+00],
        [1.3735e+03, 5.1097e+02, 1.4258e+03, 6.2867e+02, 8.6987e-01, 4.0000e+00],
        [1.1189e+03, 6.7544e+02, 1.1842e+03, 8.0109e+02, 8.6120e-01, 4.0000e+00],
 